# Phase 2+3 — Preprocessing + Training

Sets up MONAI data pipelines, builds 3D U-Net, and runs the training loop.

**Kernel:** `gtx-1080-IP`

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: k:\Coding\projects\integrative-project


## Configuration

Change `CONFIG` to `'full'` for real training.

In [2]:
import torch
from src.utils.config import load_config
from src.utils.seed import set_seed
from src.utils.logging import init_wandb, finish_wandb, print_log

CONFIG = 'dev'  # 'dev' for fast testing, 'full' for real training
RESUME_FROM = None  # Set to 'checkpoints/last_model.pth' to resume

cfg = load_config(CONFIG)
set_seed(cfg['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config: {CONFIG}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

[Seed] All seeds set to 42
Config: dev
Device: cuda
GPU: NVIDIA GeForce GTX 1080
VRAM: 8.0 GB


## Initialize WandB (optional)

In [3]:
run_name = f'brats-3dunet-{CONFIG}'
init_wandb(cfg, run_name=run_name)

[Logging] WandB disabled


## Load Data & Create Splits

In [4]:
from src.data.dataset import discover_brats_samples, get_monai_file_list
from src.data.splits import create_splits

max_samples = cfg['data'].get('max_samples')
samples = discover_brats_samples(cfg['paths']['data_root'], max_samples=max_samples)

train_samples, val_samples, test_samples = create_splits(
    samples,
    ratios=cfg['data']['split_ratios'],
    seed=cfg['seed'],
    splits_dir=cfg['paths']['splits_dir'],
)

train_files = get_monai_file_list(train_samples)
val_files = get_monai_file_list(val_samples)

print(f'Training samples: {len(train_files)}')
print(f'Validation samples: {len(val_files)}')

[Dataset] Found 5 valid samples in K:\Coding\projects\integrative-project\data\raw
[Splits] Loading existing split from K:\Coding\projects\integrative-project\data\splits\split.json
[Splits] Subset too small for saved split — re-splitting available samples
[Splits] Loaded split: train=3, val=1, test=1
Training samples: 3
Validation samples: 1


## Build Data Pipelines

In [5]:
from monai.data import CacheDataset, DataLoader, list_data_collate
from src.data.transforms import get_train_transforms, get_val_transforms

train_transforms = get_train_transforms(cfg)
val_transforms = get_val_transforms(cfg)

# Cache rate: 100% for small datasets, 50% for large
cache_rate = 1.0 if max_samples and max_samples <= 10 else 0.5

train_ds = CacheDataset(
    data=train_files,
    transform=train_transforms,
    cache_rate=cache_rate,
    num_workers=cfg['training'].get('num_workers', 2),
)

val_ds = CacheDataset(
    data=val_files,
    transform=val_transforms,
    cache_rate=1.0,
    num_workers=cfg['training'].get('num_workers', 2),
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg['training']['batch_size'],
    shuffle=True,
    num_workers=cfg['training'].get('num_workers', 2),
    collate_fn=list_data_collate,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=cfg['training'].get('num_workers', 2),
    pin_memory=True,
)

print(f'Train batches per epoch: {len(train_loader)}')
print(f'Val batches per epoch: {len(val_loader)}')

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
monai.transforms.croppad.dictionary CropForegroundd.__init__:allow_smaller: Current default value of argument `allow_smaller=True` has been deprecated since version 1.2. It will be changed to `allow_smaller=False` in version 1.5.
Loading dataset: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]

Train batches per epoch: 3
Val batches per epoch: 1


## Build Model, Loss, Optimizer, Scheduler

In [6]:
from src.models.unet3d import get_model
from src.training.losses import get_loss_function

model = get_model(cfg)
loss_fn = get_loss_function(cfg)

opt_cfg = cfg['training']['optimizer']
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=opt_cfg['lr'],
    weight_decay=opt_cfg.get('weight_decay', 1e-5),
)

sch_cfg = cfg['training']['scheduler']
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=sch_cfg['T_0'],
    T_mult=sch_cfg.get('T_mult', 2),
    eta_min=sch_cfg.get('eta_min', 1e-7),
)

[Model] 3D U-Net created
[Model]   Total parameters:     1,190,358
[Model]   Trainable parameters: 1,190,358
[Model]   Channels: [16, 32, 64, 128]
[Model]   In/Out: 4/3
[Loss] DiceCE loss (lambda_dice=1.0, lambda_ce=1.0)


## Train!

In [7]:
from src.training.trainer import Trainer

trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
)

# Resume from checkpoint if set
if RESUME_FROM:
    trainer.resume_from_checkpoint(RESUME_FROM)

history = trainer.train()

[INFO] Starting training for 5 epochs
[INFO]   Mixed precision: True
[INFO]   Early stopping: False (patience=50)
[INFO]   Device: cuda
[INFO] Epoch 1/5 | Loss: 1.3870 | Dice: WT=0.0040 TC=0.0016 ET=0.0000 Mean=0.0019 | LR: 9.99e-05 | Time: 7.7s
[INFO]   → New best model! Dice=0.0019
[INFO] Epoch 2/5 | Loss: 1.3798 | Dice: WT=0.0039 TC=0.0015 ET=0.0000 Mean=0.0018 | LR: 9.96e-05 | Time: 6.8s
[INFO] Epoch 3/5 | Loss: 1.1667 | Dice: WT=0.0039 TC=0.0014 ET=0.0000 Mean=0.0018 | LR: 9.91e-05 | Time: 7.2s
[INFO] Epoch 4/5 | Loss: 1.3459 | Dice: WT=0.0040 TC=0.0014 ET=0.0000 Mean=0.0018 | LR: 9.84e-05 | Time: 7.4s
[INFO] Epoch 5/5 | Loss: 1.1942 | Dice: WT=0.0041 TC=0.0013 ET=0.0000 Mean=0.0018 | LR: 9.76e-05 | Time: 6.9s
[INFO] Training complete. Best mean Dice: 0.0019


## Plot Training Curves

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dice curves
axes[1].plot(history['val_dice_wt'], label='WT')
axes[1].plot(history['val_dice_tc'], label='TC')
axes[1].plot(history['val_dice_et'], label='ET')
axes[1].plot(history['val_dice_mean'], label='Mean', linewidth=2, linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice')
axes[1].set_title('Validation Dice')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)
save_path = out_dir / 'training_curves.png'
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Training curves saved to {save_path}')


In [9]:
finish_wandb()

print('\n✅ Training complete.')
print(f'  Best checkpoint: {cfg["paths"]["checkpoint_dir"]}/best_model.pth')
print(f'  Last checkpoint: {cfg["paths"]["checkpoint_dir"]}/last_model.pth')


✅ Training complete.
  Best checkpoint: K:\Coding\projects\integrative-project\checkpoints/best_model.pth
  Last checkpoint: K:\Coding\projects\integrative-project\checkpoints/last_model.pth
